In [1]:
# ============================================================
# D4 — Stage 4 Validation — Branch C: Deterministic Normalisation
# 0. Imports and validation configuration
# ============================================================

from google.colab import files
from pathlib import Path
from html.parser import HTMLParser

import hashlib
import html
import json
import re
import unicodedata

import numpy as np
import pandas as pd

DOCUMENT_ID = "D4"
BRANCH_ID = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_REFERENCE_COUNT = 83
EXPECTED_HEADER_COUNT = 8
EXPECTED_CONCEPT_COUNT = 75

EXTRACTION_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted"
]

REFERENCE_FIELDS = EXTRACTION_FIELDS + ["Source Location"]

# Frozen D4 observation identity from Validation A/B.
MATCHING_FIELDS = [
    "Section",
    "Concept Name"
]

ALLOWED_PUBLICATION_FLAGS = {
    "YES",
    "NO",
    None
}

OUTPUT_DIR = Path("outputs_D4_validation_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH_ID, "-", BRANCH_NAME)
print("Expected Stage 1 records:", EXPECTED_REFERENCE_COUNT)

Document: D4
Branch: C - Deterministic normalisation
Expected Stage 1 records: 83


In [2]:
# ------------------------------------------------------------
# 1. Upload validation inputs
# ------------------------------------------------------------
# Required:
#   1) D4_reference_values.csv
#   2) D4_branch_C_parsed_extraction.json
#   3) D4_branch_C_structure_check.json
#   4) D4_branch_C_normalisation_check.json
#
# The old Validation C notebook used D4_branch_C_normalised_records.csv
# as the scoring reference. That is intentionally NOT used here.
# The Stage 1 reference remains the authoritative ground truth.

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    f for f in uploaded_files
    if f.lower().endswith(".csv")
]

json_files = [
    f for f in uploaded_files
    if f.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one CSV: D4_reference_values.csv."
    )

if len(json_files) != 3:
    raise ValueError(
        "Upload exactly three JSON files: Branch C parsed extraction, "
        "structure check, and normalisation check."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
STRUCTURE_CHECK_FILE = None
NORMALISATION_CHECK_FILE = None

for file_name in json_files:

    with open(file_name, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and "schema_validity" in obj
        and "record_structure_issues" in obj
        and "observed_record_count" in obj
    ):
        STRUCTURE_CHECK_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and obj.get("parent_branch") == PARENT_BRANCH
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_CHECK_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify D4 Branch C parsed extraction JSON."
    )

if STRUCTURE_CHECK_FILE is None:
    raise ValueError(
        "Could not identify D4 Branch C structure-check JSON."
    )

if NORMALISATION_CHECK_FILE is None:
    raise ValueError(
        "Could not identify D4 Branch C normalisation-check JSON."
    )

print("Reference:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Structure check:", STRUCTURE_CHECK_FILE)
print("Normalisation check:", NORMALISATION_CHECK_FILE)

Saving D4_branch_C_parsed_extraction.json to D4_branch_C_parsed_extraction.json
Saving D4_branch_C_normalisation_check.json to D4_branch_C_normalisation_check.json
Saving D4_branch_C_structure_check.json to D4_branch_C_structure_check.json
Saving D4_reference_values.csv to D4_reference_values.csv
Reference: D4_reference_values.csv
Parsed extraction: D4_branch_C_parsed_extraction.json
Structure check: D4_branch_C_structure_check.json
Normalisation check: D4_branch_C_normalisation_check.json


In [3]:
# ------------------------------------------------------------
# 2. Load inputs, verify identities, and preserve provenance
# ------------------------------------------------------------

with open(PARSED_EXTRACTION_FILE, "r", encoding="utf-8-sig") as f:
    extraction_json = json.load(f)

with open(STRUCTURE_CHECK_FILE, "r", encoding="utf-8-sig") as f:
    structure_check = json.load(f)

with open(NORMALISATION_CHECK_FILE, "r", encoding="utf-8-sig") as f:
    normalisation_check = json.load(f)

reference_df_raw = pd.read_csv(
    REFERENCE_FILE,
    encoding="utf-8-sig"
)

extracted_df_raw = pd.DataFrame(
    extraction_json["records"]
)

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "structure check": structure_check,
    "normalisation check": normalisation_check
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH_ID:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )

if normalisation_check.get("parent_branch") != PARENT_BRANCH:
    raise ValueError(
        "The uploaded normalisation check does not identify "
        "Branch B as the parent representation."
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256": sha256_file(PARSED_EXTRACTION_FILE),
    "structure_check_file": STRUCTURE_CHECK_FILE,
    "structure_check_sha256": sha256_file(STRUCTURE_CHECK_FILE),
    "normalisation_check_file": NORMALISATION_CHECK_FILE,
    "normalisation_check_sha256": sha256_file(NORMALISATION_CHECK_FILE)
}

print("Reference shape:", reference_df_raw.shape)
print("Extraction shape:", extracted_df_raw.shape)

Reference shape: (83, 5)
Extraction shape: (83, 4)


In [4]:
# ------------------------------------------------------------
# 3. Verify the fixed Stage 1 reference
# ------------------------------------------------------------

reference_schema_valid = (
    reference_df_raw.columns.tolist()
    == REFERENCE_FIELDS
)

reference_record_count_valid = (
    len(reference_df_raw)
    == EXPECTED_REFERENCE_COUNT
)

header_count = int(
    (
        reference_df_raw["Section"]
        == "Header"
    ).sum()
)

concept_count = int(
    len(reference_df_raw)
    - header_count
)

header_count_valid = (
    header_count
    == EXPECTED_HEADER_COUNT
)

concept_count_valid = (
    concept_count
    == EXPECTED_CONCEPT_COUNT
)

if not reference_schema_valid:
    raise ValueError(
        "The D4 Stage 1 reference schema is invalid."
    )

if not reference_record_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_REFERENCE_COUNT} Stage 1 records, "
        f"found {len(reference_df_raw)}."
    )

if not header_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_HEADER_COUNT} Header records, "
        f"found {header_count}."
    )

if not concept_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_CONCEPT_COUNT} concept records, "
        f"found {concept_count}."
    )

print("Reference schema valid:", reference_schema_valid)
print("Reference records:", len(reference_df_raw))
print("Header records:", header_count)
print("Concept records:", concept_count)

Reference schema valid: True
Reference records: 83
Header records: 8
Concept records: 75


In [5]:
# ------------------------------------------------------------
# 4. Reuse Branch C schema diagnostics
# ------------------------------------------------------------
# Record-count agreement is NOT schema validity.
# It remains a separate scope/completeness outcome.

schema_validity = bool(
    structure_check.get("schema_validity", False)
)

schema_diagnostics = {
    "valid_json":
        bool(structure_check.get("valid_json", False)),

    "top_level_object_valid":
        bool(structure_check.get("top_level_object_valid", False)),

    "document_id_present":
        bool(structure_check.get("document_id_present", False)),

    "document_id_correct":
        bool(structure_check.get("document_id_correct", False)),

    "branch_present":
        bool(structure_check.get("branch_present", False)),

    "branch_correct":
        bool(structure_check.get("branch_correct", False)),

    "records_present":
        bool(structure_check.get("records_present", False)),

    "records_is_list":
        bool(structure_check.get("records_is_list", False)),

    "records_with_structure_issues":
        int(structure_check.get("records_with_structure_issues", 0)),

    "records_with_type_issues":
        int(structure_check.get("records_with_type_issues", 0)),

    "publication_flag_issue_count":
        int(structure_check.get("publication_flag_issue_count", 0)),

    "duplicate_record_key_count":
        int(structure_check.get("duplicate_record_key_count", 0)),

    "unexpected_source_row_field_count":
        int(structure_check.get(
            "unexpected_source_row_field_count", 0
        )),

    "html_reintroduction_detected":
        bool(structure_check.get(
            "html_reintroduction_detected", False
        )),

    "html_entity_reintroduction_detected":
        bool(structure_check.get(
            "html_entity_reintroduction_detected", False
        )),

    "representation_adherence_valid":
        bool(structure_check.get(
            "representation_adherence_valid", False
        )),

    "expected_record_count":
        int(structure_check.get(
            "expected_record_count",
            EXPECTED_REFERENCE_COUNT
        )),

    "observed_record_count":
        int(structure_check.get(
            "observed_record_count",
            len(extracted_df_raw)
        )),

    "record_count_valid":
        bool(structure_check.get(
            "record_count_valid", False
        )),

    "expected_header_record_count":
        int(structure_check.get(
            "expected_header_record_count",
            EXPECTED_HEADER_COUNT
        )),

    "observed_header_record_count":
        int(structure_check.get(
            "observed_header_record_count", 0
        )),

    "header_count_valid":
        bool(structure_check.get(
            "header_count_valid", False
        )),

    "expected_concept_record_count":
        int(structure_check.get(
            "expected_concept_record_count",
            EXPECTED_CONCEPT_COUNT
        )),

    "observed_concept_record_count":
        int(structure_check.get(
            "observed_concept_record_count", 0
        )),

    "concept_count_valid":
        bool(structure_check.get(
            "concept_count_valid", False
        )),

    "scope_complete":
        bool(structure_check.get("scope_complete", False)),

    "schema_validity":
        schema_validity
}

print(json.dumps(
    schema_diagnostics,
    indent=2,
    ensure_ascii=False
))

{
  "valid_json": true,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "publication_flag_issue_count": 0,
  "duplicate_record_key_count": 0,
  "unexpected_source_row_field_count": 0,
  "html_reintroduction_detected": false,
  "html_entity_reintroduction_detected": false,
  "representation_adherence_valid": true,
  "expected_record_count": 83,
  "observed_record_count": 83,
  "record_count_valid": true,
  "expected_header_record_count": 8,
  "observed_header_record_count": 8,
  "header_count_valid": true,
  "expected_concept_record_count": 75,
  "observed_concept_record_count": 75,
  "concept_count_valid": true,
  "scope_complete": true,
  "schema_validity": true
}


In [6]:
# ------------------------------------------------------------
# 5. Reuse Branch C representation-integrity diagnostics
# ------------------------------------------------------------
# These describe the deterministic B→C transformation.
# They do not establish LLM extraction correctness.

representation_integrity = {
    "parent_branch":
        normalisation_check.get("parent_branch"),

    "parent_equivalence_passed":
        bool(normalisation_check.get(
            "parent_equivalence_passed", False
        )),

    "normalisation_integrity_passed":
        bool(normalisation_check.get(
            "normalisation_integrity_passed", False
        )),

    "all_worksheets_preserved":
        bool(normalisation_check.get(
            "all_worksheets_preserved", False
        )),

    "source_row_sequence_preserved":
        bool(normalisation_check.get(
            "source_row_sequence_preserved", False
        )),

    "target_scope_sequence_preserved":
        bool(normalisation_check.get(
            "target_scope_sequence_preserved", False
        )),

    "target_record_count_preserved":
        bool(normalisation_check.get(
            "target_record_count_preserved", False
        )),

    "fenced_block_count_preserved":
        bool(normalisation_check.get(
            "fenced_block_count_preserved", False
        )),

    "deterministic_block_transformation_verified":
        bool(normalisation_check.get(
            "deterministic_block_transformation_verified", False
        )),

    "expected_null_structure_verified":
        bool(normalisation_check.get(
            "expected_null_structure_verified", False
        )),

    "normalised_values_with_html_tags":
        int(normalisation_check.get(
            "normalised_values_with_html_tags", 0
        )),

    "normalised_values_with_html_entities":
        int(normalisation_check.get(
            "normalised_values_with_html_entities", 0
        )),

    "hyperlink_targets_preserved":
        bool(normalisation_check.get(
            "hyperlink_targets_preserved", False
        )),

    "parent_literal_null_count":
        normalisation_check.get("parent_literal_null_count"),

    "expected_branch_C_null_count":
        normalisation_check.get("expected_branch_C_null_count"),

    "observed_branch_C_null_count":
        normalisation_check.get("observed_branch_C_null_count"),

    "normalisation_generated_null_count":
        normalisation_check.get("normalisation_generated_null_count")
}

print(json.dumps(
    representation_integrity,
    indent=2,
    ensure_ascii=False
))

if not representation_integrity["normalisation_integrity_passed"]:
    print(
        "WARNING: Branch C normalisation integrity did not pass. "
        "Validation can describe the model output, but Stage 5 "
        "interpretation must distinguish preprocessing loss from "
        "extraction error."
    )

{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "all_worksheets_preserved": true,
  "source_row_sequence_preserved": true,
  "target_scope_sequence_preserved": true,
  "target_record_count_preserved": true,
  "fenced_block_count_preserved": true,
  "deterministic_block_transformation_verified": true,
  "expected_null_structure_verified": true,
  "normalised_values_with_html_tags": 0,
  "normalised_values_with_html_entities": 0,
  "hyperlink_targets_preserved": true,
  "parent_literal_null_count": 15,
  "expected_branch_C_null_count": 39,
  "observed_branch_C_null_count": 39,
  "normalisation_generated_null_count": 24
}


In [7]:
# ------------------------------------------------------------
# 6. Verify extraction fields
# ------------------------------------------------------------

missing_extraction_columns = [
    field for field in EXTRACTION_FIELDS
    if field not in extracted_df_raw.columns
]

# Preserve raw parsed extraction.
# A separate validation copy is used below.
extracted_df = extracted_df_raw.copy(deep=True)

for field in missing_extraction_columns:
    extracted_df[field] = None

extracted_df = extracted_df[
    EXTRACTION_FIELDS
].copy()

reference_df = reference_df_raw[
    REFERENCE_FIELDS
].copy()

print("Extracted records:", len(extracted_df))
print("Missing extraction columns:", missing_extraction_columns)

Extracted records: 83
Missing extraction columns: []


In [24]:
# ------------------------------------------------------------
# 7. Reproduce the pre-specified D4 Branch C normalisation
# ------------------------------------------------------------
# IMPORTANT:
# The Stage 1 reference dataset is NOT replaced.
# These functions create a comparison-only representation target
# from the fixed Stage 1 source values.
#
# They reproduce the deterministic transformations used in Branch C
# before the LLM was executed.
#
# IMPORTANT D4 DETAIL:
# HTML entities such as &nbsp; are decoded by the HTML parser into
# Unicode non-breaking spaces. Unicode-space harmonisation is therefore
# performed BOTH before and after HTML/entity decoding to reproduce the
# final Branch C representation correctly.

UNICODE_SPACE_CHARACTERS = {
    "\u00a0",  # NO-BREAK SPACE
    "\u1680",
    "\u2000",
    "\u2001",
    "\u2002",
    "\u2003",
    "\u2004",
    "\u2005",
    "\u2006",
    "\u2007",
    "\u2008",
    "\u2009",
    "\u200a",
    "\u202f",
    "\u205f",
    "\u3000"
}

APOSTROPHE_REPLACEMENTS = {
    "\u2018": "'",
    "\u2019": "'",
    "\u02bc": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "\u2010": "-",
    "\u2011": "-",
    "\u2012": "-",
    "\u2013": "-",
    "\u2014": "-",
    "\u2212": "-"
}


def harmonise_unicode_spaces(text):
    """
    Replace Unicode space characters with ordinary ASCII spaces.

    This helper is intentionally applied both before and after HTML
    parsing because entities such as &nbsp; are converted to Unicode
    non-breaking spaces only during HTML/entity decoding.
    """
    if text is None:
        return None

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    return text


class MetadataHTMLNormaliser(HTMLParser):

    BLOCK_TAGS = {
        "p",
        "div",
        "li",
        "ul",
        "ol",
        "table",
        "tr",
        "td",
        "th",
        "section",
        "article",
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6"
    }

    def __init__(self):
        super().__init__(
            convert_charrefs=True
        )

        self.parts = []
        self.active_link = None
        self.active_link_text = []

    def add_boundary(self):

        if (
            self.parts
            and self.parts[-1] != "\n"
        ):
            self.parts.append(
                "\n"
            )

    def handle_starttag(
        self,
        tag,
        attrs
    ):

        tag = tag.lower()

        attributes = dict(
            attrs
        )

        if (
            tag in self.BLOCK_TAGS
            or tag == "br"
        ):
            self.add_boundary()

        if tag == "a":

            self.active_link = (
                attributes.get("href")
            )

            self.active_link_text = []

    def handle_endtag(
        self,
        tag
    ):

        tag = tag.lower()

        if tag == "a":

            if self.active_link:

                label = "".join(
                    self.active_link_text
                ).strip()

                if label:

                    self.parts.append(
                        f" [{self.active_link}]"
                    )

                else:

                    self.parts.append(
                        self.active_link
                    )

            self.active_link = None
            self.active_link_text = []

        if tag in self.BLOCK_TAGS:
            self.add_boundary()

    def handle_data(
        self,
        data
    ):

        self.parts.append(
            data
        )

        if self.active_link is not None:

            self.active_link_text.append(
                data
            )

    def get_text(self):

        return "".join(
            self.parts
        )


def normalise_plain_text(value):
    """
    Reproduce the D4 Branch C Concept Value normalisation.

    Operations:
    - preserve nulls;
    - Unicode NFKC normalisation;
    - harmonise Unicode spaces;
    - harmonise apostrophes and dash characters;
    - recover paragraph/block boundaries from HTML;
    - retain hyperlink targets as [URL];
    - decode HTML entities;
    - harmonise Unicode spaces introduced by entity decoding;
    - collapse intra-line whitespace;
    - remove empty lines;
    - convert visually empty content to None.

    This is used only to construct a comparison copy of the fixed
    Stage 1 reference. The original reference dataset is not modified.
    """

    if pd.isna(value):
        return None

    text = str(
        value
    )

    # Preserve the Branch C convention for literal null.
    if text == "null":
        return None

    # ----------------------------------------
    # Initial Unicode normalisation
    # ----------------------------------------

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = harmonise_unicode_spaces(
        text
    )

    for source, target in (
        APOSTROPHE_REPLACEMENTS.items()
    ):

        text = text.replace(
            source,
            target
        )

    for source, target in (
        DASH_REPLACEMENTS.items()
    ):

        text = text.replace(
            source,
            target
        )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    # ----------------------------------------
    # HTML / markup normalisation
    # ----------------------------------------

    parser = MetadataHTMLNormaliser()

    try:

        parser.feed(
            text
        )

        parser.close()

        text = parser.get_text()

    except Exception:

        # Conservative deterministic fallback.
        text = html.unescape(
            text
        )

    # Decode any entities that remain after parsing.
    text = html.unescape(
        text
    )

    # ---------------------------------------------------------
    # CRITICAL FIX:
    # &nbsp; and similar entities become Unicode spaces only
    # AFTER HTML decoding. Harmonise them at this point too.
    # ---------------------------------------------------------

    text = harmonise_unicode_spaces(
        text
    )

    # Apply the same character harmonisation again in case
    # entity decoding introduced relevant Unicode characters.

    for source, target in (
        APOSTROPHE_REPLACEMENTS.items()
    ):

        text = text.replace(
            source,
            target
        )

    for source, target in (
        DASH_REPLACEMENTS.items()
    ):

        text = text.replace(
            source,
            target
        )

    # ----------------------------------------
    # Line and whitespace normalisation
    # ----------------------------------------

    normalised_lines = []

    for line in text.splitlines():

        line = re.sub(
            r"[ \t\f\v]+",
            " ",
            line
        ).strip()

        if line:

            normalised_lines.append(
                line
            )

    if not normalised_lines:
        return None

    return "\n".join(
        normalised_lines
    )


def normalise_inline_text(value):
    """
    Conservative comparison normalisation for fields that were not
    subject to HTML-rich Concept Value processing.

    Used for:
    - Section
    - Concept Name
    - Publication Restricted
    - D4 matching identity
    """

    if pd.isna(value):
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    text = harmonise_unicode_spaces(
        text
    )

    for source, target in (
        APOSTROPHE_REPLACEMENTS.items()
    ):

        text = text.replace(
            source,
            target
        )

    for source, target in (
        DASH_REPLACEMENTS.items()
    ):

        text = text.replace(
            source,
            target
        )

    text = re.sub(
        r"[ \t\f\v]+",
        " ",
        text
    ).strip()

    return (
        text
        if text
        else None
    )


def exact_null_safe(
    left,
    right
):
    """
    Exact comparison with explicit null equivalence.
    """

    left_missing = (
        left is None
        or (
            isinstance(left, float)
            and np.isnan(left)
        )
    )

    right_missing = (
        right is None
        or (
            isinstance(right, float)
            and np.isnan(right)
        )
    )

    if (
        left_missing
        and right_missing
    ):
        return True

    if (
        left_missing
        or right_missing
    ):
        return False

    return (
        left == right
    )


print(
    "D4 Branch C comparison-normalisation "
    "functions loaded successfully."
)

D4 Branch C comparison-normalisation functions loaded successfully.


In [25]:
# ------------------------------------------------------------
# 8. Build the Branch C comparison target from Stage 1
# ------------------------------------------------------------
# This is a comparison copy only.
# Source Location remains unchanged for traceability.

reference_target_df = reference_df.copy(deep=True)

reference_target_df[
    "Section"
] = reference_target_df[
    "Section"
].apply(
    normalise_inline_text
)

reference_target_df[
    "Concept Name"
] = reference_target_df[
    "Concept Name"
].apply(
    normalise_inline_text
)

reference_target_df[
    "Concept Value"
] = reference_target_df[
    "Concept Value"
].apply(
    normalise_plain_text
)

reference_target_df[
    "Publication Restricted"
] = reference_target_df[
    "Publication Restricted"
].apply(
    normalise_inline_text
)

comparison_target_diagnostics = {
    "stage1_reference_records":
        int(len(reference_df)),

    "comparison_target_records":
        int(len(reference_target_df)),

    "concept_values_changed_by_branch_C_rules":
        int(
            sum(
                not exact_null_safe(
                    source,
                    target
                )
                for source, target
                in zip(
                    reference_df["Concept Value"],
                    reference_target_df["Concept Value"]
                )
            )
        ),

    "concept_values_normalised_to_null":
        int(
            sum(
                (
                    not pd.isna(source)
                    and target is None
                )
                for source, target
                in zip(
                    reference_df["Concept Value"],
                    reference_target_df["Concept Value"]
                )
            )
        ),

    "section_values_changed":
        int(
            sum(
                not exact_null_safe(source, target)
                for source, target
                in zip(
                    reference_df["Section"],
                    reference_target_df["Section"]
                )
            )
        ),

    "concept_names_changed":
        int(
            sum(
                not exact_null_safe(source, target)
                for source, target
                in zip(
                    reference_df["Concept Name"],
                    reference_target_df["Concept Name"]
                )
            )
        ),

    "publication_flags_changed":
        int(
            sum(
                not exact_null_safe(source, target)
                for source, target
                in zip(
                    reference_df["Publication Restricted"],
                    reference_target_df["Publication Restricted"]
                )
            )
        ),

    "authoritative_ground_truth_remains_stage1":
        True,

    "separate_branch_C_reference_dataset_created":
        False
}

print(json.dumps(
    comparison_target_diagnostics,
    indent=2,
    ensure_ascii=False
))

{
  "stage1_reference_records": 83,
  "comparison_target_records": 83,
  "concept_values_changed_by_branch_C_rules": 53,
  "concept_values_normalised_to_null": 0,
  "section_values_changed": 0,
  "concept_names_changed": 0,
  "publication_flags_changed": 0,
  "authoritative_ground_truth_remains_stage1": true,
  "separate_branch_C_reference_dataset_created": false
}


In [26]:
# ------------------------------------------------------------
# 9. Create frozen D4 matching keys and handle duplicates
# ------------------------------------------------------------
# Identity normalisation is the same D4 principle defined in A/B.
# Value fields remain excluded from alignment.

ref_cmp = reference_target_df.copy(deep=True)
ext_cmp = extracted_df.copy(deep=True)

for df in (ref_cmp, ext_cmp):

    df["_matching_key"] = df.apply(
        lambda row: (
            normalise_inline_text(
                row["Section"]
            ),
            normalise_inline_text(
                row["Concept Name"]
            )
        ),
        axis=1
    )

reference_duplicate_mask = (
    ref_cmp["_matching_key"]
    .duplicated(keep=False)
)

if reference_duplicate_mask.any():
    raise ValueError(
        "The fixed Stage 1 D4 identity is not unique "
        "after the pre-specified matching normalisation."
    )

extraction_duplicate_mask = (
    ext_cmp["_matching_key"]
    .duplicated(keep="first")
)

duplicate_extracted_records = ext_cmp[
    extraction_duplicate_mask
].copy()

ext_unique = ext_cmp[
    ~extraction_duplicate_mask
].copy()

print(
    "Reference duplicate keys:",
    int(reference_duplicate_mask.sum())
)

print(
    "Additional extracted duplicate records:",
    len(duplicate_extracted_records)
)

Reference duplicate keys: 0
Additional extracted duplicate records: 0


In [27]:
# ------------------------------------------------------------
# 10. One-to-one record alignment
# ------------------------------------------------------------

validation_df = ref_cmp.merge(
    ext_unique,
    on="_matching_key",
    how="outer",
    suffixes=("_ref", "_ext"),
    indicator=True,
    validate="one_to_one"
)

aligned_mask = (
    validation_df["_merge"]
    == "both"
)

print(
    validation_df["_merge"]
    .value_counts(dropna=False)
)

_merge
both          83
left_only      0
right_only     0
Name: count, dtype: int64


In [28]:
# ------------------------------------------------------------
# 11. Compare all requested fields
# ------------------------------------------------------------
# Primary correctness = exact agreement with the deterministic
# Branch C representation target derived from Stage 1.
#
# The raw Stage 1 Concept Value is also attached later for audit.

for field in EXTRACTION_FIELDS:
    validation_df[
        f"{field}_match"
    ] = False

validation_df.loc[
    aligned_mask,
    "Section_match"
] = validation_df.loc[
    aligned_mask
].apply(
    lambda row:
        exact_null_safe(
            row["Section_ref"],
            row["Section_ext"]
        ),
    axis=1
)

validation_df.loc[
    aligned_mask,
    "Concept Name_match"
] = validation_df.loc[
    aligned_mask
].apply(
    lambda row:
        exact_null_safe(
            row["Concept Name_ref"],
            row["Concept Name_ext"]
        ),
    axis=1
)

validation_df.loc[
    aligned_mask,
    "Concept Value_match"
] = validation_df.loc[
    aligned_mask
].apply(
    lambda row:
        exact_null_safe(
            row["Concept Value_ref"],
            row["Concept Value_ext"]
        ),
    axis=1
)

validation_df.loc[
    aligned_mask,
    "Publication Restricted_match"
] = validation_df.loc[
    aligned_mask
].apply(
    lambda row:
        exact_null_safe(
            row["Publication Restricted_ref"],
            row["Publication Restricted_ext"]
        ),
    axis=1
)

PRIMARY_MATCH_COLUMNS = [
    "Section_match",
    "Concept Name_match",
    "Concept Value_match",
    "Publication Restricted_match"
]

validation_df[
    "all_fields_match"
] = (
    aligned_mask
    & validation_df[
        PRIMARY_MATCH_COLUMNS
    ].all(axis=1)
)

# Attach original Stage 1 values for traceability.
source_lookup = (
    reference_df
    .assign(
        _matching_key=lambda df:
            df.apply(
                lambda row: (
                    normalise_inline_text(
                        row["Section"]
                    ),
                    normalise_inline_text(
                        row["Concept Name"]
                    )
                ),
                axis=1
            )
    )
    .set_index("_matching_key")
)

validation_df[
    "Stage1_Source_Concept_Value"
] = validation_df[
    "_matching_key"
].map(
    source_lookup[
        "Concept Value"
    ]
)

validation_df[
    "Stage1_Source_Location"
] = validation_df[
    "_matching_key"
].map(
    source_lookup[
        "Source Location"
    ]
)

In [29]:
# ------------------------------------------------------------
# 12. Classify record outcomes
# ------------------------------------------------------------

def classify_record(row):

    if row["_merge"] == "left_only":
        return "missing"

    if row["_merge"] == "right_only":
        return "unsupported_unmatched"

    if bool(row["all_fields_match"]):
        return "fully_correct"

    return "discrepant"


validation_df[
    "record_status"
] = validation_df.apply(
    classify_record,
    axis=1
)

missing_records = validation_df[
    validation_df["record_status"]
    == "missing"
].copy()

unsupported_unmatched = validation_df[
    validation_df["record_status"]
    == "unsupported_unmatched"
].copy()

discrepant_records = validation_df[
    validation_df["record_status"]
    == "discrepant"
].copy()

fully_correct_records = validation_df[
    validation_df["record_status"]
    == "fully_correct"
].copy()

duplicate_extracted_records[
    "record_status"
] = "unsupported_duplicate"

print("Missing:", len(missing_records))
print(
    "Unsupported unmatched:",
    len(unsupported_unmatched)
)
print(
    "Unsupported duplicate extras:",
    len(duplicate_extracted_records)
)
print("Discrepant:", len(discrepant_records))
print("Fully correct:", len(fully_correct_records))

Missing: 0
Unsupported unmatched: 0
Unsupported duplicate extras: 0
Discrepant: 2
Fully correct: 81


In [30]:
# ------------------------------------------------------------
# 13. Calculate common validation metrics
# ------------------------------------------------------------

N_REF = int(
    len(reference_df)
)

N_EXT = int(
    len(extracted_df)
)

N_ALIGNED = int(
    aligned_mask.sum()
)

N_MISSING = int(
    len(missing_records)
)

N_UNSUPPORTED_UNMATCHED = int(
    len(unsupported_unmatched)
)

N_DUPLICATE_EXTRAS = int(
    len(duplicate_extracted_records)
)

N_UNSUPPORTED = (
    N_UNSUPPORTED_UNMATCHED
    + N_DUPLICATE_EXTRAS
)

N_DISCREPANT = int(
    len(discrepant_records)
)

N_CORRECT = int(
    len(fully_correct_records)
)

completeness = (
    N_ALIGNED / N_REF
    if N_REF else 0.0
)

missing_rate = (
    N_MISSING / N_REF
    if N_REF else 0.0
)

record_precision = (
    N_CORRECT / N_EXT
    if N_EXT else 0.0
)

record_recall = (
    N_CORRECT / N_REF
    if N_REF else 0.0
)

record_f1 = (
    2
    * record_precision
    * record_recall
    / (
        record_precision
        + record_recall
    )
    if (
        record_precision
        + record_recall
    )
    else 0.0
)

unsupported_rate = (
    N_UNSUPPORTED / N_EXT
    if N_EXT else 0.0
)

discrepancy_rate = (
    N_DISCREPANT / N_ALIGNED
    if N_ALIGNED else 0.0
)

aligned_df = validation_df[
    aligned_mask
].copy()

field_accuracy_among_aligned = {
    field:
        float(
            aligned_df[
                f"{field}_match"
            ].mean()
        )
        if N_ALIGNED
        else 0.0

    for field
    in EXTRACTION_FIELDS
}

correct_field_instances = int(
    aligned_df[
        PRIMARY_MATCH_COLUMNS
    ].sum().sum()
)

expected_field_instances = int(
    N_REF
    * len(PRIMARY_MATCH_COLUMNS)
)

overall_field_accuracy = (
    correct_field_instances
    / expected_field_instances
    if expected_field_instances
    else 0.0
)

print("Reference records:", N_REF)
print("Extracted records:", N_EXT)
print("Aligned records:", N_ALIGNED)
print("Fully correct:", N_CORRECT)
print("Discrepant:", N_DISCREPANT)
print("Missing:", N_MISSING)
print("Unsupported:", N_UNSUPPORTED)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1, 4))
print(
    "Overall field accuracy:",
    round(overall_field_accuracy, 4)
)

Reference records: 83
Extracted records: 83
Aligned records: 83
Fully correct: 81
Discrepant: 2
Missing: 0
Unsupported: 0
Completeness: 1.0
Exact F1: 0.9759
Overall field accuracy: 0.994


In [31]:
# ------------------------------------------------------------
# 14. Field-level error summary
# ------------------------------------------------------------

field_error_rows = []

for field in EXTRACTION_FIELDS:

    col = f"{field}_match"

    correct_aligned = int(
        aligned_df[col].sum()
    )

    incorrect_aligned = int(
        N_ALIGNED
        - correct_aligned
    )

    field_error_rows.append({
        "field":
            field,

        "used_in_matching_key":
            field in MATCHING_FIELDS,

        "aligned_records_evaluated":
            N_ALIGNED,

        "correct_values_among_aligned":
            correct_aligned,

        "incorrect_values_among_aligned":
            incorrect_aligned,

        "accuracy_among_aligned":
            (
                round(
                    correct_aligned
                    / N_ALIGNED,
                    4
                )
                if N_ALIGNED
                else 0.0
            ),

        "missing_expected_instances":
            N_MISSING,

        "overall_correct_instances":
            correct_aligned,

        "overall_expected_instances":
            N_REF,

        "overall_field_accuracy":
            (
                round(
                    correct_aligned
                    / N_REF,
                    4
                )
                if N_REF
                else 0.0
            )
    })

field_error_summary_df = pd.DataFrame(
    field_error_rows
)

display(
    field_error_summary_df
)

,field,used_in_matching_key,aligned_records_evaluated,correct_values_among_aligned,incorrect_values_among_aligned,accuracy_among_aligned,missing_expected_instances,overall_correct_instances,overall_expected_instances,overall_field_accuracy
0,Section,True,83,83,0,1.0000,0,83,83,1.0000
1,Concept Name,True,83,83,0,1.0000,0,83,83,1.0000
2,Concept Value,False,83,81,2,0.9759,0,81,83,0.9759
3,Publication Restricted,False,83,83,0,1.0000,0,83,83,1.0000


In [32]:
# ------------------------------------------------------------
# 15. D4-specific Concept Value diagnostics
# ------------------------------------------------------------
# These diagnostics help distinguish:
# - correct extraction of the intended Branch C target,
# - source values that were intentionally changed by preprocessing,
# - residual extraction discrepancies.

concept_value_diagnostics = pd.DataFrame({
    "aligned_records": [N_ALIGNED],

    "branch_C_target_exact_matches": [
        int(
            aligned_df[
                "Concept Value_match"
            ].sum()
        )
    ],

    "branch_C_target_exact_mismatches": [
        int(
            N_ALIGNED
            - aligned_df[
                "Concept Value_match"
            ].sum()
        )
    ],

    "stage1_values_changed_by_branch_C_rules": [
        comparison_target_diagnostics[
            "concept_values_changed_by_branch_C_rules"
        ]
    ],

    "stage1_values_normalised_to_null": [
        comparison_target_diagnostics[
            "concept_values_normalised_to_null"
        ]
    ],

    "html_reintroduction_detected": [
        schema_diagnostics[
            "html_reintroduction_detected"
        ]
    ],

    "html_entity_reintroduction_detected": [
        schema_diagnostics[
            "html_entity_reintroduction_detected"
        ]
    ]
})

display(
    concept_value_diagnostics
)

,aligned_records,branch_C_target_exact_matches,branch_C_target_exact_mismatches,stage1_values_changed_by_branch_C_rules,stage1_values_normalised_to_null,html_reintroduction_detected,html_entity_reintroduction_detected
0,83,81,2,53,0,False,False


In [33]:
# ------------------------------------------------------------
# 16. Build final validation summary
# ------------------------------------------------------------

summary = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "discrepant_records":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "unsupported_records":
        N_UNSUPPORTED,

    "unsupported_unmatched_records":
        N_UNSUPPORTED_UNMATCHED,

    "unsupported_duplicate_records":
        N_DUPLICATE_EXTRAS,

    "completeness":
        round(completeness, 4),

    "missing_rate":
        round(missing_rate, 4),

    "record_precision_exact":
        round(record_precision, 4),

    "record_recall_exact":
        round(record_recall, 4),

    "record_f1_exact":
        round(record_f1, 4),

    "unsupported_rate":
        round(unsupported_rate, 4),

    "discrepancy_rate_among_aligned":
        round(discrepancy_rate, 4),

    "overall_field_accuracy":
        round(overall_field_accuracy, 4),

    "field_accuracy_among_aligned": {
        key:
            round(value, 4)
        for key, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "content_evaluable":
        True,

    "schema_diagnostics":
        schema_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_key_fields":
        MATCHING_FIELDS,

    "comparison_rules_frozen_from_branch_A":
        True,

    "reference_dataset_branch_independent":
        True,

    "comparison_target": {
        "authoritative_ground_truth":
            "Fixed Stage 1 D4 reference dataset",

        "separate_branch_C_reference_dataset_used":
            False,

        "branch_C_target_derivation":
            (
                "Comparison-only deterministic transformation "
                "of Stage 1 values using the exact pre-extraction "
                "Branch C normalisation rules."
            ),

        "identity_alignment":
            (
                "Section + Concept Name after the same conservative "
                "D4 identity normalisation used in Validation A/B; "
                "value fields excluded from alignment."
            ),

        "primary_correctness":
            (
                "Exact agreement with the deterministic Branch C "
                "representation target for Section, Concept Name, "
                "Concept Value, and Publication Restricted."
            ),

        "original_stage1_values_retained_for_traceability":
            True
    },

    "comparison_target_diagnostics":
        comparison_target_diagnostics,

    "recovered_or_repaired_extraction_used":
        False,

    "reference_values_used_to_construct_branch_C_input":
        False,

    "normalisation_note":
        (
            "The Stage 1 reference dataset remains unchanged. "
            "A comparison-only target is deterministically derived "
            "using the same D4 Branch C transformation that was fixed "
            "before extraction. No model output is used to define "
            "the target and the preserved extraction is not modified."
        ),

    "input_provenance":
        input_provenance
}

print(json.dumps(
    summary,
    indent=2,
    ensure_ascii=False
))

{
  "document_id": "D4",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "reference_records": 83,
  "extracted_records": 83,
  "aligned_records": 83,
  "fully_correct_records": 81,
  "discrepant_records": 2,
  "missing_records": 0,
  "unsupported_records": 0,
  "unsupported_unmatched_records": 0,
  "unsupported_duplicate_records": 0,
  "completeness": 1.0,
  "missing_rate": 0.0,
  "record_precision_exact": 0.9759,
  "record_recall_exact": 0.9759,
  "record_f1_exact": 0.9759,
  "unsupported_rate": 0.0,
  "discrepancy_rate_among_aligned": 0.0241,
  "overall_field_accuracy": 0.994,
  "field_accuracy_among_aligned": {
    "Section": 1.0,
    "Concept Name": 1.0,
    "Concept Value": 0.9759,
    "Publication Restricted": 1.0
  },
  "schema_validity": true,
  "content_evaluable": true,
  "schema_diagnostics": {
    "valid_json": true,
    "top_level_object_valid": true,
    "document_id_present": true,
    "document_id_correct": true,
    "branch_pr

In [34]:
# ------------------------------------------------------------
# 17. Compact overall-results table
# ------------------------------------------------------------

overall_metrics_df = pd.DataFrame([
    {"metric": "Reference records", "value": N_REF},
    {"metric": "Extracted records", "value": N_EXT},
    {"metric": "Aligned records", "value": N_ALIGNED},
    {"metric": "Fully correct records", "value": N_CORRECT},
    {"metric": "Discrepant records", "value": N_DISCREPANT},
    {"metric": "Missing records", "value": N_MISSING},
    {"metric": "Unsupported records", "value": N_UNSUPPORTED},
    {"metric": "Completeness", "value": round(completeness, 4)},
    {"metric": "Exact precision", "value": round(record_precision, 4)},
    {"metric": "Exact recall", "value": round(record_recall, 4)},
    {"metric": "Exact F1", "value": round(record_f1, 4)},
    {
        "metric": "Overall field accuracy",
        "value": round(overall_field_accuracy, 4)
    },
    {
        "metric": "Unsupported rate",
        "value": round(unsupported_rate, 4)
    },
    {
        "metric": "Schema validity",
        "value": schema_validity
    },
    {
        "metric": "Branch C normalisation integrity",
        "value":
            representation_integrity[
                "normalisation_integrity_passed"
            ]
    }
])

display(
    overall_metrics_df
)

,metric,value
0,Reference records,83
1,Extracted records,83
2,Aligned records,83
3,Fully correct records,81
4,Discrepant records,2
5,Missing records,0
6,Unsupported records,0
7,Completeness,1.0
8,Exact precision,0.9759
9,Exact recall,0.9759


In [35]:
# ------------------------------------------------------------
# 18. Validation integrity checks
# ------------------------------------------------------------

assert (
    N_ALIGNED
    + N_MISSING
    == N_REF
)

assert (
    N_ALIGNED
    + N_UNSUPPORTED_UNMATCHED
    + N_DUPLICATE_EXTRAS
    == N_EXT
)

assert (
    N_CORRECT
    + N_DISCREPANT
    == N_ALIGNED
)

assert (
    int(reference_duplicate_mask.sum())
    == 0
)

for metric_name, metric_value in {
    "completeness":
        completeness,

    "missing_rate":
        missing_rate,

    "record_precision":
        record_precision,

    "record_recall":
        record_recall,

    "record_f1":
        record_f1,

    "unsupported_rate":
        unsupported_rate,

    "discrepancy_rate":
        discrepancy_rate,

    "overall_field_accuracy":
        overall_field_accuracy
}.items():

    assert (
        0.0
        <= metric_value
        <= 1.0
    ), (
        f"Invalid {metric_name}: "
        f"{metric_value}"
    )

print(
    "Validation integrity checks passed."
)

Validation integrity checks passed.


In [36]:
# ------------------------------------------------------------
# 19. Export validation artefacts
# ------------------------------------------------------------

validation_df.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_validation_detailed.csv",
    index=False,
    encoding="utf-8-sig"
)

missing_records.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_missing_records.csv",
    index=False,
    encoding="utf-8-sig"
)

unsupported_unmatched.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_unsupported_unmatched_records.csv",
    index=False,
    encoding="utf-8-sig"
)

duplicate_extracted_records.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_unsupported_duplicate_records.csv",
    index=False,
    encoding="utf-8-sig"
)

discrepant_records.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_discrepant_records.csv",
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_fully_correct_records.csv",
    index=False,
    encoding="utf-8-sig"
)

field_error_summary_df.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_field_error_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

concept_value_diagnostics.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_concept_value_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([
    comparison_target_diagnostics
]).to_csv(
    OUTPUT_DIR
    / "D4_branch_C_comparison_target_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)

overall_metrics_df.to_csv(
    OUTPUT_DIR
    / "D4_branch_C_overall_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    OUTPUT_DIR
    / "D4_branch_C_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Validation artefacts saved.")

Validation artefacts saved.


In [37]:
# ------------------------------------------------------------
# 20. Final validation report
# ------------------------------------------------------------

final_report = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "discrepant_records":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "unsupported_records":
        N_UNSUPPORTED,

    "completeness":
        round(completeness, 4),

    "record_precision_exact":
        round(record_precision, 4),

    "record_recall_exact":
        round(record_recall, 4),

    "record_f1_exact":
        round(record_f1, 4),

    "overall_field_accuracy":
        round(overall_field_accuracy, 4),

    "schema_validity":
        schema_validity,

    "normalisation_integrity_passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],

    "fixed_stage1_reference_used":
        True,

    "separate_branch_C_reference_dataset_used":
        False
}

print(json.dumps(
    final_report,
    indent=2,
    ensure_ascii=False
))

{
  "document_id": "D4",
  "branch": "C",
  "reference_records": 83,
  "extracted_records": 83,
  "aligned_records": 83,
  "fully_correct_records": 81,
  "discrepant_records": 2,
  "missing_records": 0,
  "unsupported_records": 0,
  "completeness": 1.0,
  "record_precision_exact": 0.9759,
  "record_recall_exact": 0.9759,
  "record_f1_exact": 0.9759,
  "overall_field_accuracy": 0.994,
  "schema_validity": true,
  "normalisation_integrity_passed": true,
  "fixed_stage1_reference_used": true,
  "separate_branch_C_reference_dataset_used": false
}


In [38]:
# ------------------------------------------------------------
# 21. Download validation artefacts
# ------------------------------------------------------------

for output_file in sorted(
    OUTPUT_DIR.iterdir()
):

    if output_file.is_file():

        print(
            "Downloading:",
            output_file.name
        )

        files.download(
            output_file
        )

Downloading: D4_branch_C_comparison_target_diagnostics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D4_branch_C_concept_value_diagnostics.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D4_branch_C_validation_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>